In [1]:
import os
import numpy as np
import xarray as xr
import rasterio
from rasterio.transform import xy
from rasterio.transform import from_origin

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import ListedColormap, BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

from scipy import stats
from scipy.stats import rankdata

from glob import glob

In [7]:
# Step 1: Set path to TIF files
tif_folder = "../output/Wind200"
tif_files = sorted(glob(os.path.join(tif_folder, "*.tif")))

# Step 2: Extract dates from filenames
dates = [os.path.basename(f).replace(".tif", "") for f in tif_files]  # ['DJF 1991', 'MAM 1991', ..., 'SON 2024']

# Step 3: Read one file to get spatial dimensions and transform

with rasterio.open(tif_files[0]) as src:
    height, width = src.height, src.width
    transform = src.transform
    crs = src.crs

    # Generate coordinate values from row and column indices
    row_indices = np.arange(height)
    col_indices = np.arange(width)

    # Get center coordinates
    lon, _ = xy(transform, np.zeros_like(col_indices), col_indices, offset="center")
    _, lat = xy(transform, row_indices, np.zeros_like(row_indices), offset="center")

    lon = np.array(lon)
    lat = np.array(lat)

    if lat[0] > lat[-1]:
        lat = lat[::-1]
        reverse_lat = True
    else:
        reverse_lat = False

# Ensure latitude is in ascending order (as required by xarray)
if lat[0] > lat[-1]:
    lat = lat[::-1]
    reverse_lat = True
else:
    reverse_lat = False

# Step 4: Read both bands into separate numpy arrays
u_data = []
v_data = []
for file in tif_files:
    with rasterio.open(file) as src:
        u_array = src.read(1)  # Band 1: u component
        v_array = src.read(2)  # Band 2: v component
        if reverse_lat:
            u_array = np.flipud(u_array)
            v_array = np.flipud(v_array)
        u_data.append(u_array)
        v_data.append(v_array)

u_data = np.stack(u_data)  # Shape: (n_times, height, width)
v_data = np.stack(v_data)  # Shape: (n_times, height, width)

# Step 5: Create xarray Dataset with u and v as separate variables
coords = {
    "date": dates,
    "latitude": lat,
    "longitude": lon
}
dims = ("date", "latitude", "longitude")

ds = xr.Dataset(
    {
        "u": xr.DataArray(u_data, dims=dims, coords=coords),
        "v": xr.DataArray(v_data, dims=dims, coords=coords),
    }
)

# Step 6: Save to NetCDF
ds.to_netcdf("../output/Wind200_anomalies.nc")

In [6]:
anomalies_full = xr.open_dataset("../output/Wind850_anomalies.nc", engine='netcdf4')
anomalies_full

<xarray.Dataset> Size: 229MB
Dimensions:    (date: 137, latitude: 401, longitude: 521)
Coordinates:
  * date       (date) <U8 4kB 'DJF 1991' 'DJF 1992' ... 'SON 2023' 'SON 2024'
  * latitude   (latitude) float64 3kB -20.0 -19.75 -19.5 ... 79.5 79.75 80.0
  * longitude  (longitude) float64 4kB -50.0 -49.75 -49.5 ... 79.5 79.75 80.0
Data variables:
    u          (date, latitude, longitude) float32 114MB ...
    v          (date, latitude, longitude) float32 114MB ...